# 04 · Variance reduction: CUPED and regression adjustment

**Goal:** use what was known about customers *before* the emails were sent to make the effect estimates more precise.

Randomization already makes the simple differences in notebook 03 **unbiased**. Variance reduction does not fix bias; it removes noise. Narrower confidence intervals mean the same precision with fewer customers, or the same customers detecting smaller effects (notebook 05 turns this into sample sizes).

Two methods, from simplest to broader:

1. **CUPED** with past-year spend (`history`) as the pre-experiment covariate.
2. **Regression adjustment** with all pre-treatment covariates.

In [1]:
import numpy as np
import pandas as pd

from hillstrom.data import load_customers
from hillstrom.effects import diff_means_welch
from hillstrom.plots import ARM_LABELS
from hillstrom.variance import cuped_adjust, regression_adjusted_effect

df = load_customers()
EMAILS = ["mens", "womens"]
METRICS = ["visit", "conversion", "spend"]

## 1. CUPED

CUPED (Controlled-experiment Using Pre-Experiment Data) replaces each outcome $Y$ by

$$Y^{\text{adj}} = Y - \theta\,(X - \bar X), \qquad \theta = \frac{\operatorname{cov}(Y, X)}{\operatorname{var}(X)},$$

where $X$ is measured before the experiment. Because the emails cannot change $X$, $\bar X$ is the same in expectation in every arm, so the difference in mean $Y^{\text{adj}}$ between arms estimates the same effect as before. Its variance is smaller by the factor $1 - \rho^2$, where $\rho$ is the correlation between $X$ and $Y$. $\theta$ is estimated on both arms pooled.

How much CUPED can help therefore depends entirely on $\rho$. The correlations between the pre-treatment variables and the two-week outcomes:

In [2]:
df[["history", "recency", "visit", "conversion", "spend"]].corr().loc[["history", "recency"], METRICS].style.format("{:+.3f}")

,visit,conversion,spend
history,+0.065,+0.029,+0.022
recency,-0.075,-0.024,-0.016


They are weak: $\rho = 0.065$ between `history` and `visit` means at most $\rho^2 \approx 0.4\%$ variance reduction, and less for conversion and spend. Past-year spend says little about whether a customer visits or buys in a particular two-week window, and the outcomes are mostly zeros.

In [3]:
cuped_rows = []
for email in EMAILS:
    sub = df[df["arm"].isin([email, "control"])]
    treated = (sub["arm"] == email).to_numpy()
    for metric in METRICS:
        y = sub[metric].to_numpy()
        y_adj, theta = cuped_adjust(y, sub["history"].to_numpy())
        raw = diff_means_welch(y[treated], y[~treated])
        adj = diff_means_welch(y_adj[treated], y_adj[~treated])
        cuped_rows.append(
            {
                "contrast": f"{ARM_LABELS[email]} − No E-Mail",
                "metric": metric,
                "theta": theta,
                "raw_estimate": raw.estimate,
                "cuped_estimate": adj.estimate,
                "raw_ci_width": raw.ci_high - raw.ci_low,
                "cuped_ci_width": adj.ci_high - adj.ci_low,
                "variance_reduction": 1 - np.var(y_adj, ddof=1) / np.var(y, ddof=1),
            }
        )
cuped = pd.DataFrame(cuped_rows)
cuped.style.format(
    {"theta": "{:.2e}", "raw_estimate": "{:+.4f}", "cuped_estimate": "{:+.4f}", "raw_ci_width": "{:.4f}",
     "cuped_ci_width": "{:.4f}", "variance_reduction": "{:.2%}"}
).hide(axis="index")

contrast,metric,theta,raw_estimate,cuped_estimate,raw_ci_width,cuped_ci_width,variance_reduction
Mens E-Mail − No E-Mail,visit,9.13e-05,+0.0766,+0.0764,0.0133,0.0132,0.44%
Mens E-Mail − No E-Mail,conversion,1.10e-05,+0.0068,+0.0068,0.0036,0.0036,0.09%
Mens E-Mail − No E-Mail,spend,1.22e-03,+0.7698,+0.7674,0.5694,0.5693,0.04%
Womens E-Mail − No E-Mail,visit,8.35e-05,+0.0452,+0.0451,0.0127,0.0127,0.40%
Womens E-Mail − No E-Mail,conversion,8.72e-06,+0.0031,+0.0031,0.0032,0.0032,0.07%
Womens E-Mail − No E-Mail,spend,1.03e-03,+0.4244,+0.4227,0.5109,0.5108,0.04%


**Result:** CUPED with past-year spend reduces variance by less than half a percent for visits and by hundredths of a percent for spend. The estimates barely move (as they should, since randomization already balanced `history`), and the intervals are essentially unchanged.